# 02 Feature activations at "="

Step 1 of the label plan (preregistration section 3): collect every transcoder feature's activation at the "=" position.

Setup: Accelerator GPU T4 x2, Internet on, secret `HF_TOKEN`.

**New session:** run cells 1 to 4 in order. If a later cell complains about the `transformers` version, restart once and continue from cell 2.

**After pushing new code:** run `update()` (cell 3). No restart, and the model stays loaded. Use modules as `activations.eq_activations(...)`, not `from ... import eq_activations`, so the reload reaches them.

In [1]:
# 1. get the code and the libraries
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3 pytest

Cloning into 'helix_addition'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 128 (delta 55), reused 94 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 168.70 KiB | 3.12 MiB/s, done.
Resolving deltas: 100% (55/55), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipykernel==6.17.1, but you have ipykernel 6.31.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatib

In [2]:
# 2. make helixadd importable (no install needed) and log in to Hugging Face
import sys
sys.path.insert(0, "helix_addition/src")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [3]:
# 3. update(): pull the latest code and reload every helixadd module in place
import importlib, subprocess

def update():
    print(subprocess.run(["git", "-C", "helix_addition", "pull"], capture_output=True, text=True).stdout)
    names = sorted(m for m in sys.modules if m.startswith("helixadd"))
    # modules import each other, so a reload can fail until its dependencies are reloaded;
    # retry the failed ones until everything reloads
    for _ in range(len(names)):
        failed = []
        for name in names:
            try:
                importlib.reload(sys.modules[name])
            except Exception as e:
                failed.append((name, e))
        if not failed:
            print("reloaded", names)
            return
        names = [n for n, _ in failed]
    raise RuntimeError(f"could not reload: {failed}")

!cd helix_addition && python -m pytest -q

................                                                         [100%]
16 passed in 22.29s


In [4]:
# 4. load the model (once per session)
from helixadd import model as hmodel, prompts, activations
model = hmodel.load_model()

config.yaml:   0%|          | 0.00/140 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

layer_15.safetensors:   0%|          | 0.00/1.08G [00:00<?, ?B/s]

layer_10.safetensors:   0%|          | 0.00/1.08G [00:00<?, ?B/s]

layer_1.safetensors:   0%|          | 0.00/1.08G [00:00<?, ?B/s]

layer_12.safetensors:   0%|          | 0.00/1.08G [00:00<?, ?B/s]

... (13 outputs left out)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer


## How many features fire at "="?

In [5]:
_, acts = model.get_activations(prompts.make_prompt(36, 59))
print(acts.shape)
eq = acts[:, -1, :]                     # every layer, last position ("="), every feature
print((eq > 0).sum(dim=1))              # active features per layer
print("total:", (eq > 0).sum().item())

torch.Size([16, 44, 131072])
tensor([ 27,  11,  45,  59,  50,  35,  55,  34,  38,  31,  26,  58,  84, 107,
        114,  54], device='cuda:0')
total: 828


## eq_activations on one prompt

In [6]:
layers, features, values = activations.eq_activations(model, prompts.make_prompt(36, 59))
print(len(layers), layers[:5], features[:5], values[:5])

828 tensor([0, 0, 0, 0, 0]) tensor([ 1203,  4051, 10535, 15770, 19322]) tensor([0.1703, 0.2014, 0.0986, 0.1754, 0.0925])


## Test: 20 prompts

Checks that the loop and the save work before the 30-minute run. Writes to a separate test folder.

In [7]:
test = activations.collect(model, "helix_addition/results/eq_test", n=20)
print({k: v.shape for k, v in test.items()})
print("entries per prompt:", len(test["layer"]) / 20)

0 of 20
{'prompt': (14038,), 'layer': (14038,), 'feature': (14038,), 'value': (14038,), 'a': (20,), 'b': (20,)}
entries per prompt: 701.9


## Full run: all 9,992 prompts (about 30 minutes)

Run as Save Version, Save & Run All. Output: `helix_addition/results/eq_activations/`.

In [8]:
data = activations.collect(model, "helix_addition/results/eq_activations")
print({k: v.shape for k, v in data.items()})
print("entries per prompt:", len(data["layer"]) / len(data["a"]))

0 of 9992
1000 of 9992
2000 of 9992
3000 of 9992
4000 of 9992
5000 of 9992
6000 of 9992
7000 of 9992
8000 of 9992
9000 of 9992
{'prompt': (7546419,), 'layer': (7546419,), 'feature': (7546419,), 'value': (7546419,), 'a': (9992,), 'b': (9992,)}
entries per prompt: 755.246096877502
